# 🚀 3D Zack D. Films Animation — High Speed GPU Renderer
This Google Colab notebook renders your 10-second (300 frames, 480p, 30 FPS) Zack D. Films style 3D animation using a **free NVIDIA T4 GPU** in under 3 minutes.

### Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.
2. Upload `zack_animation.blend` to the files panel on the left (or let cell 2 download it).
3. Run the cells in order!

In [ ]:
# Step 1: Verify NVIDIA GPU
!nvidia-smi

In [ ]:
# Step 2: Install Blender and FFmpeg
!apt-get update -qq
!apt-get install -y blender ffmpeg > /dev/null
!blender --version

In [ ]:
# Step 3: Create Python script to enable GPU CUDA / OptiX in Cycles
gpu_config_script = '''
import bpy
prefs = bpy.context.preferences
cprefs = prefs.addons['cycles'].preferences
cprefs.get_devices()

# Prefer OPTIX, fallback to CUDA
backend = 'OPTIX' if 'OPTIX' in [d.type for d in cprefs.devices] else 'CUDA'
cprefs.compute_device_type = backend
for device in cprefs.devices:
    device.use = True
print(f'Cycles GPU Acceleration active using: {backend}')
'''
with open('set_gpu.py', 'w') as f:
    f.write(gpu_config_script)

In [ ]:
# Step 4: Render All 300 Frames on GPU
!mkdir -p output
!blender -b zack_animation.blend -P set_gpu.py -E CYCLES -s 1 -e 300 -o "//output/frame_####" -a

In [ ]:
# Step 5: Encode to MP4 (30 FPS, H.264)
!ffmpeg -framerate 30 -pattern_type glob -i 'output/*.png' -c:v libx264 -pix_fmt yuv420p zack_animation_10s.mp4

# Step 6: Download the finished video
from google.colab import files
files.download('zack_animation_10s.mp4')